# HTP 데이터셋 전처리 (AI Hub JSON → YOLO)

이 노트북은 **AI Hub HTP(House-Tree-Person) JSON 어노테이션을 YOLO 형식으로 변환하고, House / Tree / Person 데이터셋을 생성하는 전처리 워크플로우**입니다.

- AI Hub HTP 데이터의 bbox JSON을 YOLO `.txt` 포맷으로 변환
- House / Tree / Person 세 카테고리로 분리하여 독립 데이터셋 구성
- 남자사람 + 여자사람을 `male_person` / `female_person` 클래스로 병합
- 각 데이터셋의 YAML 설정 파일 생성

| 단계 | 내용 |
|------|------|
| STEP 1 | Drive 마운트 & 경로 설정 |
| STEP 2 | House / Tree / Person 클래스 매핑 정의 |
| STEP 3 | JSON → YOLO txt 변환 함수 |
| STEP 4 | House / Tree 데이터셋 변환 |
| STEP 5 | Person 데이터셋 변환 (남자사람 / 여자사람) |
| STEP 6 | Person 데이터셋 병합 (남자 + 여자 → person) |
| STEP 7 | house.yaml / tree.yaml / person.yaml 생성 |
| STEP 8 | 라벨 개수 및 클래스 분포 확인 |

## STEP 1. Drive 마운트 & 경로 설정

`PROJECT_ROOT` 를 본인의 Google Drive 내 프로젝트 폴더 경로로 변경하세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# ★ 본인의 Google Drive 프로젝트 폴더 경로로 변경하세요
BASE = '/content/drive/MyDrive/PROJECT_ROOT'

SPLIT_CONFIG = {
    'train': {
        'image_root': f'{BASE}/train/원천데이터',
        'label_root': f'{BASE}/train/라벨링데이터',
        'img_prefix': 'TS',
        'lbl_prefix': 'TL',
    },
    'val': {
        'image_root': f'{BASE}/val/원천데이터',
        'label_root': f'{BASE}/val/라벨링데이터',
        'img_prefix': 'VS',
        'lbl_prefix': 'VL',
    },
}

print('=== 경로 확인 ===')
all_ok = True
for split, cfg in SPLIT_CONFIG.items():
    for key in ['image_root', 'label_root']:
        p = cfg[key]
        exists = os.path.exists(p)
        mark = '✅' if exists else '❌ 없음'
        print(f'  {mark}  [{split}] {p}')
        if not exists:
            all_ok = False
print()
print('✅ 모든 경로 확인 완료!' if all_ok else '❌ BASE 경로를 확인해주세요')

## STEP 2. 클래스 매핑 정의

House / Tree / Person 각 카테고리별 라벨 → 클래스 ID 매핑입니다.

**Person 신발 클래스 처리 방침**
- 남자사람 / 여자사람 JSON에 `운동화`, `여자구두`, `남자구두`가 모두 등장
- 변환 단계에서는 동일 PERSON_LABEL2ID 적용 (cid 0~18)
- 병합 단계에서 `사람전체(0)` → male=`0(male_person)`, female=`1(female_person)` 로 분기하고 나머지는 +1 shift
- 최종 person 클래스: `sneakers(17)`, `female_shoes(18)`, `male_shoes(19)`

In [ ]:
# House / Tree 클래스 (enumerate 순서 = class ID)
CATEGORY_CONFIG = {
    '나무': [
        '나무전체', '기둥', '수관', '가지', '뿌리', '나뭇잎',
        '꽃', '열매', '그네', '새', '다람쥐', '구름', '달', '별',
    ],
    '집': [
        '집전체', '지붕', '집벽', '문', '창문', '굴뚝', '연기',
        '울타리', '길', '연못', '산', '나무', '잔디', '태양',
    ],
}

# Person: 남자사람 / 여자사람 JSON에 동일 매핑 적용
# 병합 단계에서 사람전체(0) → male_person(0) / female_person(1) 으로 분기
PERSON_LABEL2ID = {
    '사람전체': 0,
    '머리':     1, '얼굴': 2, '눈': 3, '코': 4, '입': 5,
    '귀':       6, '머리카락': 7, '목': 8, '상체': 9,
    '팔':      10, '손': 11, '다리': 12, '발': 13,
    '단추':    14, '주머니': 15,
    '운동화':  16, '여자구두': 17, '남자구두': 18,
}

LABEL2ID = {
    **{cat: {label: idx for idx, label in enumerate(labels)}
       for cat, labels in CATEGORY_CONFIG.items()},
    '남자사람': PERSON_LABEL2ID,
    '여자사람': PERSON_LABEL2ID,
}

for cat, mapping in LABEL2ID.items():
    print(f'[{cat}] {len(mapping)}개 클래스: {list(mapping.keys())[:4]} ...')

## STEP 3. JSON → YOLO txt 변환 함수

- `parse_resolution`: JSON meta의 `img_resolution` 문자열 파싱 (`'1280x1280'` → `(1280, 1280)`)
- `xywh_to_yolo`: 픽셀 bbox → YOLO 정규화 좌표 (0~1 clamp 포함)
- `convert_category`: 카테고리 단위 일괄 변환 (JSON → txt + 이미지 복사)

In [ ]:
import json
import shutil
from pathlib import Path

OUTPUT_ROOT = Path(f'{BASE}/HTP_DATASETS')


def parse_resolution(s):
    """'1280x1280' → (1280, 1280)"""
    w, h = s.strip().split('x')
    return int(w), int(h)


def xywh_to_yolo(x, y, w, h, iw, ih):
    """픽셀 bbox → YOLO 정규화 좌표 (cx cy bw bh)"""
    cx = min(max((x + w / 2) / iw, 0.0), 1.0)
    cy = min(max((y + h / 2) / ih, 0.0), 1.0)
    bw = min(max(w / iw, 0.001), 1.0)
    bh = min(max(h / ih, 0.001), 1.0)
    return cx, cy, bw, bh


def convert_category(split, cfg, category):
    print(f'\n=== [{split}] {category} ===')

    lp = cfg['lbl_prefix']
    ip = cfg['img_prefix']
    label_root = Path(cfg['label_root'])
    image_root = Path(cfg['image_root'])

    json_dir  = label_root / f'{lp}_{category}'
    image_dir = image_root / f'{ip}_{category}'

    out_label_dir = OUTPUT_ROOT / category / 'labels' / split
    out_image_dir = OUTPUT_ROOT / category / 'images' / split
    out_label_dir.mkdir(parents=True, exist_ok=True)
    out_image_dir.mkdir(parents=True, exist_ok=True)

    json_files = sorted(json_dir.glob('*.json'))
    label2id   = LABEL2ID[category]
    ok = err = 0

    for idx, jp in enumerate(json_files):
        try:
            with open(jp, 'r', encoding='utf-8') as f:
                data = json.load(f)

            iw, ih = parse_resolution(
                data['meta'].get('img_resolution', '1280x1280'))

            yolo_lines = []
            for bbox in data['annotations'].get('bbox', []):
                label = bbox['label']
                if label not in label2id:
                    continue
                cid = label2id[label]
                cx, cy, bw, bh = xywh_to_yolo(
                    bbox['x'], bbox['y'], bbox['w'], bbox['h'], iw, ih)
                yolo_lines.append(
                    f'{cid} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')

            (out_label_dir / f'{jp.stem}.txt').write_text(
                '\n'.join(yolo_lines), encoding='utf-8')

            image_name = Path(data['meta']['img_path']).name
            img_path   = image_dir / image_name
            if img_path.exists():
                shutil.copy(img_path, out_image_dir / image_name)
                ok += 1
            else:
                err += 1

            if (idx + 1) % 500 == 0:
                print(f'  진행 {idx+1:,}/{len(json_files):,} | 성공 {ok:,} | 에러 {err:,}')

        except Exception as e:
            print(f'  [ERROR] {jp.name}: {e}')
            err += 1

    print(f'  완료: {ok:,} / 에러: {err:,}')

## STEP 4. House / Tree 데이터셋 변환

In [ ]:
for split, cfg in SPLIT_CONFIG.items():
    for category in ['나무', '집']:
        convert_category(split, cfg, category)

print('\n✅ House / Tree 변환 완료')

## STEP 5. Person 데이터셋 변환 (남자사람 / 여자사람)

남자사람, 여자사람 JSON을 각각 별도 폴더에 변환합니다.  
이 단계의 클래스 ID는 `PERSON_LABEL2ID` 기준이며, 아직 병합 전입니다.

In [ ]:
for split, cfg in SPLIT_CONFIG.items():
    for category in ['남자사람', '여자사람']:
        convert_category(split, cfg, category)

print('\n✅ 남자사람 / 여자사람 변환 완료')

## STEP 6. Person 데이터셋 병합 (남자 + 여자 → person)

병합 시 클래스 ID 재매핑 규칙:

| 원본 cid | 남자사람 → | 여자사람 → | 의미 |
|----------|-----------|-----------|------|
| 0 (사람전체) | **0** (male_person) | **1** (female_person) | 성별 분리 |
| 1~18 (나머지 부위) | cid + 1 | cid + 1 | shift |

In [ ]:
PERSON_ROOT = OUTPUT_ROOT / 'person'

for split in ['train', 'val']:
    (PERSON_ROOT / f'labels/{split}').mkdir(parents=True, exist_ok=True)
    (PERSON_ROOT / f'images/{split}').mkdir(parents=True, exist_ok=True)


def _remap_labels(txt_path, gender):
    """gender='male' → 사람전체(0)→0, 'female' → 사람전체(0)→1, 나머지 +1"""
    lines = txt_path.read_text(encoding='utf-8').strip().splitlines()
    new_lines = []
    for line in lines:
        if not line.strip():
            continue
        parts = line.split()
        cid = int(parts[0])
        new_cid = (0 if gender == 'male' else 1) if cid == 0 else cid + 1
        parts[0] = str(new_cid)
        new_lines.append(' '.join(parts))
    return '\n'.join(new_lines)


def merge_person_dataset(split):
    print(f'\n=== {split} person merge ===')

    dst_label_dir = PERSON_ROOT / 'labels' / split
    dst_image_dir = PERSON_ROOT / 'images' / split

    for gender, src_cat in [('male', '남자사람'), ('female', '여자사람')]:
        src_label_dir = OUTPUT_ROOT / src_cat / 'labels' / split
        src_image_dir = OUTPUT_ROOT / src_cat / 'images' / split

        txts = list(src_label_dir.glob('*.txt'))
        for txt_path in txts:
            remapped = _remap_labels(txt_path, gender)
            (dst_label_dir / txt_path.name).write_text(remapped, encoding='utf-8')

        for img in src_image_dir.glob('*'):
            shutil.copy(img, dst_image_dir / img.name)

        print(f'  {gender}: {len(txts):,}개')

    print(f'  ✅ {split} merge 완료')


merge_person_dataset('train')
merge_person_dataset('val')

print('\n✅ person dataset 생성 완료')

## STEP 7. YAML 파일 생성

YOLOv8 학습에 사용할 `house.yaml`, `tree.yaml`, `person.yaml` 을 각 데이터셋 폴더에 저장합니다.  
`path` 는 실제 데이터셋 위치로 자동 설정됩니다.

In [ ]:
from pathlib import Path

DATASET_ROOT = f'{BASE}/HTP_DATASETS'

YAML_CONTENTS = {
    f'{DATASET_ROOT}/집/house.yaml': f"""path: {DATASET_ROOT}/집
train: images/train
val: images/val
nc: 14
names:
  0: house
  1: roof
  2: wall
  3: door
  4: window
  5: chimney
  6: smoke
  7: fence
  8: road
  9: pond
  10: mountain
  11: tree
  12: grass
  13: sun
""",
    f'{DATASET_ROOT}/나무/tree.yaml': f"""path: {DATASET_ROOT}/나무
train: images/train
val: images/val
nc: 14
names:
  0: tree
  1: trunk
  2: crown
  3: branch
  4: root
  5: leaf
  6: flower
  7: fruit
  8: swing
  9: bird
  10: squirrel
  11: cloud
  12: moon
  13: star
""",
    f'{DATASET_ROOT}/person/person.yaml': f"""path: {DATASET_ROOT}/person
train: images/train
val: images/val
nc: 20
names:
  0: male_person
  1: female_person
  2: head
  3: face
  4: eye
  5: nose
  6: mouth
  7: ear
  8: hair
  9: neck
  10: upper_body
  11: arm
  12: hand
  13: leg
  14: foot
  15: button
  16: pocket
  17: sneakers
  18: female_shoes
  19: male_shoes
""",
}

for path, content in YAML_CONTENTS.items():
    Path(path).write_text(content, encoding='utf-8')
    print(f'✅ {Path(path).name} 생성 완료')

print('\n✅ 모든 yaml 생성 완료')

## STEP 8. 라벨 개수 및 클래스 분포 확인

In [ ]:
from collections import Counter
from pathlib import Path

# --- 데이터셋별 파일 수 ---
print('=== 데이터셋 파일 수 ===')
for cat in ['집', '나무', 'person']:
    cat_root = OUTPUT_ROOT / cat
    for split in ['train', 'val']:
        imgs = len(list((cat_root / f'images/{split}').glob('*')))
        lbls = len(list((cat_root / f'labels/{split}').glob('*.txt')))
        print(f'  [{cat}/{split}]  이미지: {imgs:,}  라벨: {lbls:,}')
    print()

# --- person train 클래스 분포 ---
PERSON_CLASS_NAMES = {
    0: 'male_person', 1: 'female_person', 2: 'head', 3: 'face',
    4: 'eye', 5: 'nose', 6: 'mouth', 7: 'ear', 8: 'hair', 9: 'neck',
    10: 'upper_body', 11: 'arm', 12: 'hand', 13: 'leg', 14: 'foot',
    15: 'button', 16: 'pocket', 17: 'sneakers', 18: 'female_shoes',
    19: 'male_shoes',
}

print('=== person train 클래스 분포 ===')
counter = Counter()
for txt in (OUTPUT_ROOT / 'person/labels/train').glob('*.txt'):
    for line in txt.read_text(encoding='utf-8').splitlines():
        if line.strip():
            counter[int(line.split()[0])] += 1

total = sum(counter.values())
print(f'  {"ID":<4} {"클래스":<16} {"bbox수":>8}  분포')
print('  ' + '-' * 50)
for cid in sorted(counter):
    name  = PERSON_CLASS_NAMES.get(cid, f'unknown_{cid}')
    count = counter[cid]
    pct   = count / total * 100
    bar   = '█' * int(pct / 2)
    print(f'  {cid:<4} {name:<16} {count:>8,}개  {bar} {pct:.1f}%')
print(f'\n  총 bbox: {total:,}개 / 클래스: {len(counter)}개')